In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "WHQ_H.xpt"
csv_file = DATA_DIR / "WHQ_H_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2013-2014 WHQ_H XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WHD010",
    "WHD020",
    "WHQ030",
    "WHQ040",
    "WHD050",
    "WHQ060",
    "WHQ070",
    "WHD080A",
    "WHD080B",
    "WHD080C",
    "WHD080D",
    "WHD080E",
    "WHD080F",
    "WHD080G",
    "WHD080H",
    "WHD080I",
    "WHD080J",
    "WHD080K",
    "WHD080M",
    "WHD080N",
    "WHD080O",
    "WHD080P",
    "WHD080Q",
    "WHD080R",
    "WHD080S",
    "WHD080T",
    "WHD080U",
    "WHD080L",
    "WHD110",
    "WHD120",
    "WHD130",
    "WHD140",
    "WHQ150"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "WHQ_H column names/order differ from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 6464:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 34:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,464 rows x 34 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Direct inspection of attached WHQ_H:
#
#     genuine decimal fields = NONE
#     string fields          = NONE
#
# All fields contain whole-number values only.
#
#
# This includes self-reported:
#
#     height in inches
#     weight in pounds
#     age
#
# CDC standardized metric responses into whole-number
# inches/pounds in the released WHD variables.
#
#
# IMPORTANT:
#
# This applies ONLY to this WHQ_H dataset.
#
# It does NOT change your overall rule:
#
#     Dietary decimals -> keep AS IS
#     Lab decimals     -> keep AS IS
#     Exam decimals    -> keep AS IS
#     Sample weights   -> keep AS IS
#
# Never globally round true decimal variables.
#

DECIMAL_COLUMNS = []

STRING_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(INTEGER_COLUMNS) != 34:

    raise ValueError(
        "Expected exactly 34 whole-number variables."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "String fields:",
    STRING_COLUMNS
)

print(
    "Whole-number fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT artifact:
#
#     5.397605346934028e-79
#
# The attached WHQ_H contains NONE.
#
# Therefore DO NOT perform any tiny -> zero replacement.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 34 WHQ_H fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in WHQ_H."
    )


if true_zero_total > 0:

    print(
        "\nOrdinary zeros by variable:"
    )

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected ordinary numeric zeros "
        "found in WHQ_H."
    )


print(
    "PASS: WHQ_H contains no tiny artifacts "
    "and no ordinary numeric zeros."
)


# ============================================================
# 6. NO ZERO CORRECTION REQUIRED
# ============================================================

#
# Unlike some earlier WHQ cycles:
#
#     no tiny value occurs
#
# Therefore:
#
#     no value replacement is performed
#
# expected_df is simply the original XPT.
#

expected_df = source_df.copy()


print(
    "\nPASS: No zero restoration required."
)


# ============================================================
# 7. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# Run BEFORE converting fields to Int64.
#
# np.round() is used ONLY to determine whether a
# number is whole-number.
#
# It does NOT modify the dataframe.
#

print(
    "\n--- GENUINE DECIMAL DETECTION ---"
)


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )

        print(
            f"{col}: "
            f"{fractional_count:,} fractional observations"
        )


if fractional_columns:

    raise ValueError(
        "STOP: Unexpected genuine decimal fields detected.\n"
        "They will NOT be rounded or converted to Int64:\n"
        + str(
            fractional_columns
        )
    )


print(
    "PASS: No genuine fractional variables "
    "across all 34 fields."
)


# ============================================================
# 8. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(
        seqn_values
    ),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 73557:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 83729:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN 73557-83729 validated."
)


# ============================================================
# 9. CONVERT PROVEN WHOLE-NUMBER FIELDS TO Int64
# ============================================================

#
# All 34 fields were proven whole-number.
#
# NO .round() is used during conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(
                values
            ),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 34 WHQ_H fields converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 10. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )


    print(
        "Observed:",
        actual_values
    )


    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 11. HELPER - RANGE/SPECIAL CODE CHECK
# ============================================================

def check_range_special(
    column,
    low,
    high,
    expected_range_count,
    expected_missing,
    special_counts=None
):

    if special_counts is None:
        special_counts = {}


    s = df[column]


    range_count = int(
        s.between(
            low,
            high
        )
        .sum()
    )


    missing_count = int(
        s.isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        f"Range {low}-{high}:",
        f"{range_count:,}"
    )

    print(
        "Missing:",
        f"{missing_count:,}"
    )


    if range_count != expected_range_count:

        raise ValueError(
            f"{column}: range count differs.\n"
            f"Expected: {expected_range_count:,}\n"
            f"Actual:   {range_count:,}"
        )


    if missing_count != expected_missing:

        raise ValueError(
            f"{column}: missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {missing_count:,}"
        )


    for code, expected_count in special_counts.items():

        actual_count = int(
            s.eq(code)
            .sum()
        )


        print(
            f"Code {code}:",
            actual_count
        )


        if actual_count != expected_count:

            raise ValueError(
                f"{column}: code {code} count differs.\n"
                f"Expected: {expected_count:,}\n"
                f"Actual:   {actual_count:,}"
            )


    print(
        f"PASS: {column} range/special codes validated."
    )


# ============================================================
# 12. CDC CHECK - WHD010
# ============================================================

#
# Current self-reported height in inches.
#
# CDC:
#
# 48-81 = 6,376
# 7777  = 1 Refused
# 9999  = 62 Don't know
# Missing = 25
#

check_range_special(
    "WHD010",
    low=48,
    high=81,
    expected_range_count=6376,
    expected_missing=25,
    special_counts={
        7777: 1,
        9999: 62
    }
)


# ============================================================
# 13. CDC CHECK - WHD020
# ============================================================

#
# Current self-reported weight in pounds.
#

check_range_special(
    "WHD020",
    low=75,
    high=493,
    expected_range_count=6377,
    expected_missing=34,
    special_counts={
        7777: 7,
        9999: 46
    }
)


# ============================================================
# 14. CDC CHECK - WHQ030
# ============================================================

check_frequency(
    "WHQ030",
    {
        1: 3218,
        2: 369,
        3: 2865,
        9: 12
    },
    0
)


# ============================================================
# 15. CDC CHECK - WHQ040
# ============================================================

check_frequency(
    "WHQ040",
    {
        1: 581,
        2: 3806,
        3: 2070,
        7: 1,
        9: 6
    },
    0
)


# ============================================================
# 16. CDC CHECK - WHD050
# ============================================================

#
# Self-reported weight one year ago.
#

check_range_special(
    "WHD050",
    low=75,
    high=559,
    expected_range_count=6330,
    expected_missing=42,
    special_counts={
        7777: 10,
        9999: 82
    }
)


# ============================================================
# 17. CDC CHECK - WHQ060
# ============================================================

#
# Was weight change intentional?
#

check_frequency(
    "WHQ060",
    {
        1: 788,
        2: 450,
        9: 1
    },
    5225
)


# ============================================================
# 18. CDC CHECK - WHQ070
# ============================================================

#
# Tried to lose weight during previous 12 months.
#

check_frequency(
    "WHQ070",
    {
        1: 2049,
        2: 3623,
        9: 2
    },
    790
)


# ============================================================
# 19. WEIGHT-LOSS METHOD FIELD CHECK
# ============================================================

#
# WHD080A - WHD080L are "CODE ALL THAT APPLY" fields.
#
# Each field contains its own fixed code when selected,
# otherwise missing.
#
# CDC counts:
#

METHOD_FIELDS = {
    "WHD080A": (10, 1693),
    "WHD080B": (11, 883),
    "WHD080C": (12, 802),
    "WHD080D": (13, 1830),
    "WHD080E": (14, 399),
    "WHD080F": (15, 233),
    "WHD080G": (16, 115),
    "WHD080H": (17, 114),
    "WHD080I": (31, 84),
    "WHD080J": (32, 200),
    "WHD080K": (33, 26),
    "WHD080M": (34, 1033),
    "WHD080N": (30, 175),
    "WHD080O": (41, 689),
    "WHD080P": (42, 19),
    "WHD080Q": (43, 1121),
    "WHD080R": (44, 1022),
    "WHD080S": (45, 943),
    "WHD080T": (46, 996),
    "WHD080U": (35, 15),
    "WHD080L": (40, 29)
}


print(
    "\n--- WEIGHT LOSS METHOD FIELDS ---"
)


for col, (
    expected_code,
    expected_count
) in METHOD_FIELDS.items():

    s = df[col]


    observed_nonmissing = int(
        s.notna()
        .sum()
    )


    observed_code_count = int(
        s.eq(
            expected_code
        )
        .sum()
    )


    unexpected_values = (
        s.dropna()
        .loc[
            ~s.dropna()
            .eq(
                expected_code
            )
        ]
    )


    print(
        f"{col}: "
        f"code={expected_code}, "
        f"count={observed_code_count:,}"
    )


    if observed_nonmissing != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} nonmissing records, "
            f"found {observed_nonmissing:,}."
        )


    if observed_code_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} observations "
            f"with code {expected_code}."
        )


    if len(unexpected_values) != 0:

        raise ValueError(
            f"{col} contains unexpected codes."
        )


print(
    "PASS: All 21 weight-loss method fields "
    "match CDC."
)


# ============================================================
# 20. CDC CHECK - WHD110
# ============================================================

#
# Self-reported weight 10 years ago.
#
# Age >=36.
#

check_range_special(
    "WHD110",
    low=75,
    high=600,
    expected_range_count=4075,
    expected_missing=2285,
    special_counts={
        7777: 8,
        9999: 96
    }
)


# ============================================================
# 21. CDC CHECK - WHD120
# ============================================================

#
# Self-reported weight at age 25.
#

check_range_special(
    "WHD120",
    low=55,
    high=600,
    expected_range_count=4811,
    expected_missing=1440,
    special_counts={
        7777: 4,
        9999: 209
    }
)


# ============================================================
# 22. CDC CHECK - WHD130
# ============================================================

#
# Self-reported height at age 25.
#

check_range_special(
    "WHD130",
    low=50,
    high=81,
    expected_range_count=2678,
    expected_missing=3699,
    special_counts={
        7777: 2,
        9999: 85
    }
)


# ============================================================
# 23. CDC CHECK - WHD140
# ============================================================

#
# Greatest lifetime self-reported weight.
#

check_range_special(
    "WHD140",
    low=85,
    high=600,
    expected_range_count=6025,
    expected_missing=361,
    special_counts={
        7777: 9,
        9999: 69
    }
)


# ============================================================
# 24. CDC CHECK - WHQ150
# ============================================================

#
# Age when participant weighed the most.
#
# CDC:
#
# 10-79 = 5,887
# 80    = 101
# 99999 = 32
# Missing = 444
#
# CDC notes that all responses age 80+
# were recoded to 80.
#

check_range_special(
    "WHQ150",
    low=10,
    high=79,
    expected_range_count=5887,
    expected_missing=444,
    special_counts={
        80: 101,
        77777: 0,
        99999: 32
    }
)


# ============================================================
# 25. CONFIRM NO ZERO VALUES CREATED
# ============================================================

#
# Original XPT contained:
#
# tiny artifacts = 0
# ordinary zeros = 0
#
# Int64 conversion must not create zero.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO CHECK ---"
)


print(
    final_zero_counts
)


if final_zero_counts:

    raise ValueError(
        "Unexpected zero values were created:\n"
        + str(
            final_zero_counts
        )
    )


print(
    "PASS: No zero values created."
)


# ============================================================
# 26. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# WHQ_H requires NO numeric repair.
#
# Therefore all values must remain numerically identical
# to the original XPT.
#
#
# Examples:
#
#     73557.0 -> 73557
#     165.0   -> 165
#     7777.0  -> 7777
#
# These are formatting changes only.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            fidelity_failures
        )
    )


print(
    "PASS: All 34 variables preserve "
    "source XPT numeric values exactly."
)


# ============================================================
# 27. EXPORT CSV
# ============================================================

#
# CRITICAL FIDELITY RULE:
#
# DO NOT:
#
#     round()
#
# DO NOT:
#
#     float_format=
#
#
# WHQ_H itself contains no genuine decimal fields,
# but this remains the correct export pattern for
# your broader NHANES workflow.
#
# Dietary/Lab/Exam true decimals therefore stay AS IS.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 28. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 29. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT XPT -> CSV NUMERIC FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(
            csv_failures
        )
    )


print(
    "PASS: All 34 CSV variables round-trip "
    "to exact source XPT values."
)


# ============================================================
# 30. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 31. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 32. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(
            TINY_VALUE
        )
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 33. RAW CSV .0 CHECK
# ============================================================

#
# WHQ_H has NO genuine decimal variables.
#
# Therefore checking all 34 fields for ".0"
# is safe for THIS specific file.
#
#
# Examples:
#
# SEQN
#     73557.0 -> 73557
#
# WHD020
#     180.0 -> 180
#
# WHD130
#     65.0 -> 65
#
# WHQ150
#     45.0 -> 45
#
#
# DO NOT apply this global rule to files with
# genuine decimal measurements.
#

print(
    "\n--- RAW CSV .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' "
    "in any WHQ_H field."
)


# ============================================================
# 34. FINAL CSV ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO CHECK ---"
)


print(
    csv_zero_counts
)


if csv_zero_counts:

    raise ValueError(
        "Unexpected zero values found "
        "in final CSV:\n"
        + str(
            csv_zero_counts
        )
    )


print(
    "PASS: Final CSV contains no zero values."
)


# ============================================================
# 35. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 36. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 73557:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 83729:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 37. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)


print(
    "WHQ_H CONVERSION COMPLETE"
)


print(
    "=" * 90
)


print(
    "Cycle: 2013-2014"
)


print(
    "Component: Questionnaire - Weight History"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Original ordinary zeros:",
    true_zero_total
)


print(
    "Validated zero restorations: NONE"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "String variables: NONE"
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    "WHQ030 overweight:",
    int(
        df["WHQ030"]
        .eq(1)
        .sum()
    )
)


print(
    "WHQ070 tried to lose weight:",
    int(
        df["WHQ070"]
        .eq(1)
        .sum()
    )
)


print(
    "WHD080D exercised to lose weight:",
    int(
        df["WHD080D"]
        .eq(13)
        .sum()
    )
)


print(
    "WHD080Q ate more fruits/vegetables:",
    int(
        df["WHD080Q"]
        .eq(43)
        .sum()
    )
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary '.0': "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2013-2014 WHQ_H XPT -> CSV
Rows: 6,464
Columns: 34
PASS: Structure = 6,464 rows x 34 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: []
String fields: []
Whole-number fields: 34

--- ORIGINAL XPT ZERO CHECK ---
Tiny XPORT artifacts: 0
Ordinary numeric zeros: 0
PASS: WHQ_H contains no tiny artifacts and no ordinary numeric zeros.

PASS: No zero restoration required.

--- GENUINE DECIMAL DETECTION ---
PASS: No genuine fractional variables across all 34 fields.

--- SEQN ---
Minimum: 73557
Maximum: 83729
Missing: 0
Duplicates: 0
PASS: SEQN 73557-83729 validated.

PASS: All 34 WHQ_H fields converted to nullable Int64 without rounding.

--- WHD010 ---
Range 48-81: 6,376
Missing: 25
Code 7777: 1
Code 9999: 62
PASS: WHD010 range/special codes validated.

--- WHD020 ---
Range 75-493: 6,377
Missing: 34
Code 7777: 7
Code 9999: 46
PASS: WHD020 range/special codes validated.

--- WHQ030 ---
Observed: {np.int64(1): 3218, np.int64(2): 369, np.int64(3): 2865, np.int64(9): 12}
Missing